In [1]:
import pandas as pd  
from pandas import Series, DataFrame 
import uproot 
from scipy import stats
from scipy.optimize import curve_fit
from scipy.special import comb
from scipy.stats import chi2
from scipy.special import comb
from scipy.optimize import lsq_linear
import sys
from plot_tools import *
from customStats import *
#import tools
import common_tools
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
# from selection_cuts import selection_nominal
import mplhep as hep
from sklearn.model_selection import train_test_split
plt.style.use(hep.style.CMS)
plt.rcParams['figure.figsize'] = [10,8]
plt.rcParams['font.size'] = 24
plt.figure()
plt.close()
plt.rcParams.update({'figure.figsize':[10,8]})
plt.rcParams.update({'font.size':24})
import tensorflow as tf
import math
import zfit
from zfit import z
import xgboost as xgb
from scipy.interpolate import make_interp_spline
# from loadCutXGB import load_and_cutXGBclfs
from scipy.special import comb
from scipy.optimize import lsq_linear
zfit.settings.set_verbosity(0)
import json
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Oculta los mensajes de INFO y WARNING
from PDFs import *
from utils_efficiency import *
from utils_fits import *

2026-03-17 00:09:35.225607: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 00:09:35.251910: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-17 00:09:35.663579: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/ghcp/miniconda3/envs/haza_wokr_env/lib/python3.8/site-packages/zfit/__init__.py:63: UserWarning: TensorFlow warnings are by default suppressed by zfit. In order to show them, set the environment variable ZFIT_DISABLE_TF_WARNINGS=0. In order to suppress th

#  CARGA DE DATOS

In [2]:
import uproot
import pandas as pd

# --- RUTAS DE ARCHIVOS ---
f_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged.root"
f_gen_filtered = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/GenLevel_Angular_Merged_Filtered.root"
f_reco_gen = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/angular/efficiencies/datasets/RecoGenV2_Angular_Merged.root"  
x_gboost_cut = "/home/ghcp/Documentos/CINVESTAV/ANALISYS_B0tomumuKstar/BdtoK0smumu-20251110T171511Z-1-001/MyReweiting/ResultsB0_2022/AntiRadVeto_MC_NoRes_2022_Era1_v0_XGBoost_fom_cut_BDT.root"

vars_gen_to_load = ["gen_cosThetaK", "gen_cosThetaL", "gen_phi", "q2Gen"]
vars_reco_to_load = ["CosThetaK_best", "CosThetaL_best", "Phi_best", "massJ"] 
vars_xgboost_to_load = ["CosThetaK", "CosThetaL", "Phi", "massB_test", "massJ", "TotalWeight"] 

# --- CARGA DE DATOS ---
#Gen NO filt
genNFtr = uproot.open(f_gen)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"1. Gen Non-Filtered (genNFtr) cargado: {len(genNFtr)} eventos")
# Gen Filtered
genFtr = uproot.open(f_gen_filtered)['ntuple'].arrays(vars_gen_to_load, library='pd')
print(f"2. Gen Filtered (genFtr) cargado: {len(genFtr)} eventos")
# Reco Gen Level
recoGen = uproot.open(f_reco_gen)['ntuple'].arrays(vars_reco_to_load, library='pd')
print(f"3. Reco Gen Level Denom (recoGen) cargado: {len(recoGen)} eventos")
# Final selection 
recoFtr = uproot.open(x_gboost_cut)['treeBd'].arrays(vars_xgboost_to_load, library='pd')
print(f"4. Reco Final (recoFtr) cargado: {len(recoFtr)} eventos")



1. Gen Non-Filtered (genNFtr) cargado: 11589148 eventos
2. Gen Filtered (genFtr) cargado: 307688 eventos
3. Reco Gen Level Denom (recoGen) cargado: 6298017 eventos
4. Reco Final (recoFtr) cargado: 900424 eventos


In [3]:

recoFtr["q2"] = recoFtr["massJ"]**2 
recoGen["q2Gen"] = recoGen["massJ"]**2  

GenNFlt = genNFtr.copy()     
GenFlt  = genFtr.copy()       

RecoGenFlt = recoGen.copy()             
mask_mass = (recoFtr["massB_test"] > 5.0) & (recoFtr["massB_test"] < 5.6)
Reco = recoFtr[mask_mass].copy()
#2
#232 for bin7 
eff_Gen, obs_Gen = train_test_split(GenNFlt, test_size=0.1, random_state=232)
eff_GenFtr, obs_GenFtr = train_test_split(GenFlt, test_size=0.1, random_state=22)
eff_RecoGenFtr, obs_RecoGenFtr = train_test_split(RecoGenFlt, test_size=0.1, random_state=22)
eff_RecoFtr, obs_RecoFtr = train_test_split(Reco, test_size=0.1, random_state=22)

a1 = np.array(obs_Gen["gen_cosThetaL"])
a2 = np.array(obs_Gen["gen_cosThetaK"])
a3 = np.array(obs_Gen["gen_phi"])

angles = np.array([a1, a2, a3])
valid_observations_mask = ~np.isnan(angles).any(axis=0)
filtered_data = angles[:, valid_observations_mask]

In [4]:
print("Numero de eventes GEN",len(obs_Gen))
print("Numero de eventes GEN Filtered",len(obs_GenFtr))
print("Numero de eventes Reco Gen filtered",len(obs_RecoGenFtr))
print("Numero de eventes Reco Filtered",len(obs_RecoFtr))


Numero de eventes GEN 1158915
Numero de eventes GEN Filtered 30769
Numero de eventes Reco Gen filtered 629802
Numero de eventes Reco Filtered 90043


# Gen  FIT Tranformed space

In [5]:

# #%%capture data_gen_transformed_fit
# import random 
# for i in range(0,10):

#     recoFtr["q2"] = recoFtr["massJ"]**2 
#     recoGen["q2Gen"] = recoGen["massJ"]**2  

#     GenNFlt = genNFtr.copy()     
#     GenFlt  = genFtr.copy()       

#     RecoGenFlt = recoGen.copy()             
#     mask_mass = (recoFtr["massB_test"] > 5.0) & (recoFtr["massB_test"] < 5.6)
#     Reco = recoFtr[mask_mass].copy()
#     #2
#     #232 for bin7 
#     random_state=random.randint(1, 100)
#     eff_Gen, obs_Gen = train_test_split(GenNFlt, test_size=0.1, random_state=random_state)
#     eff_GenFtr, obs_GenFtr = train_test_split(GenFlt, test_size=0.1, random_state=22)
#     eff_RecoGenFtr, obs_RecoGenFtr = train_test_split(RecoGenFlt, test_size=0.1, random_state=22)
#     eff_RecoFtr, obs_RecoFtr = train_test_split(Reco, test_size=0.1, random_state=22)

#     a1 = np.array(obs_Gen["gen_cosThetaL"])
#     a2 = np.array(obs_Gen["gen_cosThetaK"])
#     a3 = np.array(obs_Gen["gen_phi"])

#     angles = np.array([a1, a2, a3])
#     valid_observations_mask = ~np.isnan(angles).any(axis=0)
#     filtered_data = angles[:, valid_observations_mask]
#     #q2_bins = {"bin1":[1.1, 2.0],"bin2": [2.0, 4.0],"bin3":[4.0, 6.0], "bin4":[6.0, 7.0], "bin5":[7.0, 8.0],"bin7":[11.0, 12.5], "bin9":[15.0, 17.0], "bin10":[17.0, 23.0]}
#     q2_bins = { "bin10":[17.0, 23.0]}
#     for binN in q2_bins.keys():
#         print(f"\n{'='*60}\nProcesando {binN}, semilla {random_state} con rango q2: {q2_bins[binN]}\n{'='*60}")
#         init_rFL, init_rS3 =0.5, 0.0
#         init_rS9, init_rAFB = 0.0, 0.0
#         init_rS4, init_rS7 = 0.0, 0.0
#         init_rS5, init_rS8 = 0.0, 0.0



#         # json_path = f"fit_results/gen_phy_slsqp/{binN}/fit_results_gen_physical_slsqp_{binN}.json"
#         # if os.path.exists(json_path):
#         #     print(f"-> Leyendo semillas iniciales de: {json_path}")
#         #     with open(json_path, 'r') as file:
#         #         data = json.load(file)
#         #     params_dict = data.get("parameters", {})
#         #     phys_params_list = [params_dict["FL"]["value"], params_dict["S3"]["value"], params_dict["S9"]["value"], params_dict["AFB"]["value"], params_dict["S4"]["value"], params_dict["S7"]["value"], params_dict["S5"]["value"], params_dict["S8"]["value"]]
#         #     inv_vals = get_inverse_values(phys_params_list)
#         #     print(inv_vals)
#         #     init_rFL, init_rS3, init_rS9, init_rAFB, init_rS4, init_rS7, init_rS5, init_rS8 = inv_vals
    
#         # ======================================================
#         # CONFIGURACIÓN DEL ESPACIO 
#         # ======================================================
#         cos_l = zfit.Space('cos_l', limits=(-1, 1))
#         cos_k = zfit.Space('cos_k', limits=(-1, 1))
#         phi   = zfit.Space('phi',   limits=(-np.pi, np.pi)) 
#         obs_ang = cos_l * cos_k * phi  

#         lim_val = 5
#         rFL  = zfit.Parameter(f'rFL_{binN}_{random_state}',  init_rFL,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS3  = zfit.Parameter(f'rS3_{binN}_{random_state}',  init_rS3, step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS9  = zfit.Parameter(f'rS9_{binN}_{random_state}',  init_rS9, step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rAFB = zfit.Parameter(f'rAFB_{binN}_{random_state}', init_rAFB, step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS4  = zfit.Parameter(f'rS4_{binN}_{random_state}',  init_rS4, step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS7  = zfit.Parameter(f'rS7_{binN}_{random_state}',  init_rS7, step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS5  = zfit.Parameter(f'rS5_{binN}_{random_state}',  init_rS5, step_size=1e-3,lower_limit=-lim_val, upper_limit=lim_val)
#         rS8  = zfit.Parameter(f'rS8_{binN}_{random_state}',  init_rS8, step_size=1e-3,lower_limit=-lim_val, upper_limit=lim_val)




#         # rFL  = zfit.Parameter(f'rFL_{binN}',  init_rFL,  step_size=1e-3)
#         # rS3  = zfit.Parameter(f'rS3_{binN}',  init_rS3, step_size=1e-3)
#         # rS9  = zfit.Parameter(f'rS9_{binN}',  init_rS9, step_size=1e-3)
#         # rAFB = zfit.Parameter(f'rAFB_{binN}', init_rAFB, step_size=1e-3)
#         # rS4  = zfit.Parameter(f'rS4_{binN}',  init_rS4, step_size=1e-3)
#         # rS7  = zfit.Parameter(f'rS7_{binN}',  init_rS7, step_size=1e-3)
#         # rS5  = zfit.Parameter(f'rS5_{binN}',  init_rS5, step_size=1e-3)
#         # rS8  = zfit.Parameter(f'rS8_{binN}',  init_rS8, step_size=1e-3)


#         # r_keys = ['rFL', 'rS3', 'rS9', 'rAFB', 'rS4', 'rS7', 'rS5', 'rS8']
#         fit_params_list = [rFL, rS3, rS9, rAFB, rS4, rS7, rS5, rS8]

#         # ======================================================
#         # CONSTRUCCIÓN DE PDFs Y CARGA DE DATOS
#         # ======================================================
#         # PDF  Transformada
#         pdf_ang_trans = FullAngular_Transformed_PDF(obs_ang, rFL, rS3, rS9, rAFB, rS4, rS7, rS5, rS8)
#         # Carga de Datos
#         obs_Gen_q2 = select_q2_bin(obs_Gen, binN, "q2Gen")
#         data_true = zfit.Data.from_numpy(array=obs_Gen_q2[["gen_cosThetaL", "gen_cosThetaK", "gen_phi"]].to_numpy(), obs=obs_ang)
#         print(f"NUMERO DE EVENTOS {len(binN)} \n")

#         # ======================================================
#         # FITS
#         # ======================================================
#         print("\n" + "="*60)
#         print(">>> INICIANDO FIT GEN LEVEL ")
#         print("="*60)
#         result_gen, errors_gen = run_fit(pdf_ang_trans, data_true)
#         print(result_gen.params)
#         results_gen_transformed_save = save_fit_results(result_gen, binN, base_dir="fit_results/gen_transformed", name=f"fit_results_gen_physical_{binN}")
#         # Extraer valores centrales y la matriz de covarianza de zfit
#         r_values = np.array([result_gen.params[p]['value'] for p in fit_params_list])
#         cov_fit = result_gen.covariance(params=fit_params_list)

#         # ======================================================
#         # PROPAGACIÓN DE ERRORES
#         # ======================================================
#         obs_order = ['FL', 'AFB', 'S3', 'S4', 'S5', 'S7', 'S8', 'S9']
#         f_transform = lambda x: get_physical_array(obs_order=obs_order, r_array=x)
#         J = compute_jacobian(f_transform, r_values)
#         cov_phys = J @ cov_fit @ J.T
#         errors_phys = np.sqrt(np.diag(cov_phys))
#         phys_vals_gen_dict = apply_transformation_equations(*r_values)
#         print("\n" + "-"*80)
#         print(f"RESUMEN DE OBSERVABLES FÍSICOS (Bin: {binN})")
#         print("-"*80)
#         print(f"{'Observable':<10} | {'Valor Físico':<15} | {'Error':<25}")
#         print("-"*80)

#         for i, key in enumerate(obs_order):
#             val = phys_vals_gen_dict.get(key, 0.0)
#             err = errors_phys[i]
#             print(f"{key:<10} | {val:>15.6f} | +/- {err:<15.6f}")
#         print("-"*80 + "\n")
        
#         # =============
#         # SAVE INFO
#         # =============
#         # save_phy_results(result_zfit=result_gen, phys_dict=phys_vals_gen_dict, cov_phys=cov_phys, obs_order=obs_order, bin_n=binN, base_dir="fit_results/gen_transformed", name=f"fit_results_gen_transformed_phys_{binN}")
#         #plot_nll_profiles(result_gen, fit_params_list, binN,base_dir="plots/profiles_transform_v1")
#         #save_correlation_matrix(result_gen, fit_params_list, binN, out_dir=f"plots/correlation_matrix/transformation_1")


# import random
# import gc
# import zfit
# import numpy as np

# # ======================================================
# # 1. OPERACIONES ESTÁTICAS FUERA DEL BUCLE
# # ======================================================
# recoFtr["q2"] = recoFtr["massJ"]**2 
# recoGen["q2Gen"] = recoGen["massJ"]**2  

# GenNFlt = genNFtr.copy()     
# GenFlt  = genFtr.copy()       
# RecoGenFlt = recoGen.copy()             

# mask_mass = (recoFtr["massB_test"] > 5.0) & (recoFtr["massB_test"] < 5.6)
# Reco = recoFtr[mask_mass].copy()

# cos_l = zfit.Space('cos_l', limits=(-1, 1))
# cos_k = zfit.Space('cos_k', limits=(-1, 1))
# phi   = zfit.Space('phi',   limits=(-np.pi, np.pi)) 
# obs_ang = cos_l * cos_k * phi  

# q2_bins = {"bin9":[15.0, 17.0], "bin10":[17.0, 23.0]}

# # ======================================================
# # PRE-GENERACIÓN DE SEMILLAS
# # ======================================================
# random_seeds = random.sample(range(1001), 100)
# print(f"Lista de semillas a procesar: {random_seeds}")

# # ======================================================
# # 2. INICIO DEL BUCLE
# # ======================================================
# for i, random_state in enumerate(random_seeds):
    
#     zfit.run.clear_graph_cache()
#     gc.collect()
    
#     for binN in q2_bins.keys():
#         print(f"\n{'='*60}\nIteración: {i} | Procesando {binN}, semilla {random_state} con rango q2: {q2_bins[binN]}\n{'='*60}")
        
#         eff_Gen, obs_Gen = train_test_split(GenNFlt, test_size=0.03, random_state=random_state)
#         obs_Gen = obs_Gen[(obs_Gen['q2Gen'] != 0)].copy()
        
#         eff_GenFtr, obs_GenFtr = train_test_split(GenFlt, test_size=0.03,random_state=22)
#         eff_RecoGenFtr, obs_RecoGenFtr = train_test_split(RecoGenFlt, test_size=0.1, random_state=22)
#         eff_RecoFtr, obs_RecoFtr = train_test_split(Reco, test_size=0.1, random_state=22)

#         a1 = np.array(obs_Gen["gen_cosThetaL"])
#         a2 = np.array(obs_Gen["gen_cosThetaK"])
#         a3 = np.array(obs_Gen["gen_phi"])

#         angles = np.array([a1, a2, a3])
#         valid_observations_mask = ~np.isnan(angles).any(axis=0)
#         filtered_data = angles[:, valid_observations_mask]
        
#         # init_rFL, init_rS3 = 0.5, 0.0
#         # init_rS9, init_rAFB = 0.0, 0.0
#         # init_rS4, init_rS7 = 0.0, 0.0
#         # init_rS5, init_rS8 = 0.0, 0.0
#         json_path = f"fit_results/gen_phy_slsqp/{binN}/fit_results_gen_physical_slsqp_{binN}.json"
#         print(f"-> Leyendo semillas iniciales de: {json_path}")
#         with open(json_path, 'r') as file:
#             data = json.load(file)
#         params_dict = data.get("parameters", {})
#         phys_params_list = [params_dict["FL"]["value"], params_dict["S3"]["value"], params_dict["S9"]["value"], params_dict["AFB"]["value"], params_dict["S4"]["value"], params_dict["S7"]["value"], params_dict["S5"]["value"], params_dict["S8"]["value"]]
#         inv_vals = get_inverse_values(phys_params_list)
#         print(inv_vals)
#         init_rFL, init_rS3, init_rS9, init_rAFB, init_rS4, init_rS7, init_rS5, init_rS8 = inv_vals
   

#         lim_val = 5
        
#         rFL  = zfit.Parameter(f'rFL_{binN}_iter{i}',  init_rFL,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS3  = zfit.Parameter(f'rS3_{binN}_iter{i}',  init_rS3,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS9  = zfit.Parameter(f'rS9_{binN}_iter{i}',  init_rS9,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rAFB = zfit.Parameter(f'rAFB_{binN}_iter{i}', init_rAFB, step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS4  = zfit.Parameter(f'rS4_{binN}_iter{i}',  init_rS4,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS7  = zfit.Parameter(f'rS7_{binN}_iter{i}',  init_rS7,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS5  = zfit.Parameter(f'rS5_{binN}_iter{i}',  init_rS5,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
#         rS8  = zfit.Parameter(f'rS8_{binN}_iter{i}',  init_rS8,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)

#         fit_params_list = [rFL, rS3, rS9, rAFB, rS4, rS7, rS5, rS8]

#         pdf_ang_trans = FullAngular_Transformed_PDF(obs_ang, rFL, rS3, rS9, rAFB, rS4, rS7, rS5, rS8)
        
#         obs_Gen_q2 = select_q2_bin(obs_Gen, binN, "q2Gen")
#         data_true = zfit.Data.from_numpy(array=obs_Gen_q2[["gen_cosThetaL", "gen_cosThetaK", "gen_phi"]].to_numpy(), obs=obs_ang)
        
#         print(f"NUMERO DE EVENTOS: {len(obs_Gen_q2)} \n") 

#         print("\n" + "="*60)
#         print(">>> INICIANDO FIT GEN LEVEL ")
#         print("="*60)
        
#         # Ejecutamos el fit (se asume que errors_gen contiene los resultados de MINOS)
#         result_gen, errors_gen = run_fit(pdf_ang_trans, data_true)
        
#         # ======================================================
#         # ¡NUEVO! COMPROBACIÓN DE ERRORES (MINOS < 1.0)
#         # ======================================================
#         # Verificamos si la estimación de errores fue exitosa y si todos los errores son menores a 1.0
#         all_errors_valid = True
        
#         if errors_gen:
#             for param in fit_params_list:
#                 # Comprobamos si el parámetro está en el diccionario de errores
#                 if param in errors_gen:
#                     param_errs = errors_gen[param]
                    
#                     # Obtenemos los valores absolutos de los errores inferior y superior
#                     lower_err = abs(param_errs.get('lower', 999.0))
#                     upper_err = abs(param_errs.get('upper', 999.0))
                    
#                     # Si alguno es mayor o igual a 1, o si la estimación falló (indicado por el 999), descartamos
#                     if lower_err >= 1.0 or upper_err >= 1.0:
#                         all_errors_valid = False
#                         print(f"  -> Descartado: {param.name} tiene errores altos (Lower: {lower_err:.4f}, Upper: {upper_err:.4f})")
#                         break # No necesitamos revisar los demás
#                 else:
#                     # Si MINOS falló para un parámetro y no devolvió nada, también lo descartamos
#                     all_errors_valid = False
#                     print(f"  -> Descartado: No se encontraron errores MINOS para {param.name}")
#                     break
#         else:
#             all_errors_valid = False
#             print("  -> Descartado: La rutina de cálculo de errores falló completamente.")

#         # Si NO todos los errores son < 1, saltamos al siguiente bin/iteración
#         if not all_errors_valid:
#             print(f">>> Saltando impresión y guardado para {binN} en iteración {i} debido a errores grandes.\n")
#             continue 
            
#         # Si llegamos aquí, significa que TODOS los errores MINOS son menores a 1.0
#         print(f">>> Ajuste exitoso con errores pequeños. Procesando y guardando resultados...\n")
        
#         print(result_gen.params)
#         results_gen_transformed_save = save_fit_results(result_gen, binN, base_dir="fit_results/gen_transformed", name=f"fit_results_gen_physical_{binN}")
        
#         r_values = np.array([result_gen.params[p]['value'] for p in fit_params_list])
#         cov_fit = result_gen.covariance(params=fit_params_list)

#         obs_order = ['FL', 'AFB', 'S3', 'S4', 'S5', 'S7', 'S8', 'S9']
#         f_transform = lambda x: get_physical_array(obs_order=obs_order, r_array=x)
#         J = compute_jacobian(f_transform, r_values)
#         cov_phys = J @ cov_fit @ J.T
#         errors_phys = np.sqrt(np.diag(cov_phys))
#         phys_vals_gen_dict = apply_transformation_equations(*r_values)
        
#         print("\n" + "-"*80)
#         print(f"RESUMEN DE OBSERVABLES FÍSICOS (Bin: {binN})")
#         print("-"*80)
#         print(f"{'Observable':<10} | {'Valor Físico':<15} | {'Error':<25}")
#         print("-"*80)

#         for j, key in enumerate(obs_order):
#             val = phys_vals_gen_dict.get(key, 0.0)
#             err = errors_phys[j]
#             print(f"{key:<10} | {val:>15.6f} | +/- {err:<15.6f}")
#         print("-"*80 + "\n")





import random
import gc
import zfit
import numpy as np

# ======================================================
# 1. OPERACIONES ESTÁTICAS FUERA DEL BUCLE
# ======================================================
recoFtr["q2"] = recoFtr["massJ"]**2 
recoGen["q2Gen"] = recoGen["massJ"]**2  

GenNFlt = genNFtr.copy()     
GenFlt  = genFtr.copy()       
RecoGenFlt = recoGen.copy()             

mask_mass = (recoFtr["massB_test"] > 5.0) & (recoFtr["massB_test"] < 5.6)
Reco = recoFtr[mask_mass].copy()

cos_l = zfit.Space('cos_l', limits=(-1, 1))
cos_k = zfit.Space('cos_k', limits=(-1, 1))
phi   = zfit.Space('phi',   limits=(-np.pi, np.pi)) 
obs_ang = cos_l * cos_k * phi  

q2_bins = {"bin10":[17.0, 23.0]}
# q2_bins = {"bin1":[1.1, 2.0],"bin2": [2.0, 4.0],"bin3":[4.0, 6.0], "bin4":[6.0, 7.0], "bin5":[7.0, 8.0],"bin7":[11.0, 12.5], "bin9":[15.0, 17.0], "bin10":[17.0, 23.0]}

# ======================================================
# PRE-GENERACIÓN DE SEMILLAS
# ======================================================
random_seeds = random.sample(range(1001), 1000)
print(f"Lista de semillas a procesar: {random_seeds}")

# ======================================================
# 2. INICIO DEL BUCLE
# ======================================================
for i, random_state in enumerate(random_seeds):
    
    zfit.run.clear_graph_cache()
    gc.collect()
    
    # --------------------------------------------------
    # VARIABLES DE CONTROL PARA LA ITERACIÓN ACTUAL
    # --------------------------------------------------
    iteration_results = {}
    iteration_valid = True
    
    for binN in q2_bins.keys():
        print(f"\n{'='*60}\nIteración: {i} | Procesando {binN}, semilla {random_state} con rango q2: {q2_bins[binN]}\n{'='*60}")
        
        eff_Gen, obs_Gen = train_test_split(GenNFlt, test_size=0.03, random_state=random_state)
        obs_Gen = obs_Gen[(obs_Gen['q2Gen'] != 0)].copy()
        
        eff_GenFtr, obs_GenFtr = train_test_split(GenFlt, test_size=0.03,random_state=22)
        eff_RecoGenFtr, obs_RecoGenFtr = train_test_split(RecoGenFlt, test_size=0.1, random_state=22)
        eff_RecoFtr, obs_RecoFtr = train_test_split(Reco, test_size=0.1, random_state=22)

        a1 = np.array(obs_Gen["gen_cosThetaL"])
        a2 = np.array(obs_Gen["gen_cosThetaK"])
        a3 = np.array(obs_Gen["gen_phi"])

        angles = np.array([a1, a2, a3])
        valid_observations_mask = ~np.isnan(angles).any(axis=0)
        filtered_data = angles[:, valid_observations_mask]
        
        json_path = f"fit_results/gen/gen_phy_slsqp/{binN}/fit_results_gen_physical_slsqp_{binN}.json"
        print(f"-> Leyendo semillas iniciales de: {json_path}")
        with open(json_path, 'r') as file:
            data = json.load(file)
        params_dict = data.get("parameters", {})
        phys_params_list = [params_dict["FL"]["value"], params_dict["S3"]["value"], params_dict["S9"]["value"], params_dict["AFB"]["value"], params_dict["S4"]["value"], params_dict["S7"]["value"], params_dict["S5"]["value"], params_dict["S8"]["value"]]
        inv_vals = get_inverse_values(phys_params_list)
        init_rFL, init_rS3, init_rS9, init_rAFB, init_rS4, init_rS7, init_rS5, init_rS8 = inv_vals
   
        lim_val = 5
        
        rFL  = zfit.Parameter(f'rFL_{binN}_iter{i}',  init_rFL,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
        rS3  = zfit.Parameter(f'rS3_{binN}_iter{i}',  init_rS3,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
        rS9  = zfit.Parameter(f'rS9_{binN}_iter{i}',  init_rS9,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
        rAFB = zfit.Parameter(f'rAFB_{binN}_iter{i}', init_rAFB, step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
        rS4  = zfit.Parameter(f'rS4_{binN}_iter{i}',  init_rS4,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
        rS7  = zfit.Parameter(f'rS7_{binN}_iter{i}',  init_rS7,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
        rS5  = zfit.Parameter(f'rS5_{binN}_iter{i}',  init_rS5,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)
        rS8  = zfit.Parameter(f'rS8_{binN}_iter{i}',  init_rS8,  step_size=1e-3, lower_limit=-lim_val, upper_limit=lim_val)

        fit_params_list = [rFL, rS3, rS9, rAFB, rS4, rS7, rS5, rS8]

        pdf_ang_trans = FullAngular_Transformed_PDF(obs_ang, rFL, rS3, rS9, rAFB, rS4, rS7, rS5, rS8)
        
        obs_Gen_q2 = select_q2_bin(obs_Gen, binN, "q2Gen")
        data_true = zfit.Data.from_numpy(array=obs_Gen_q2[["gen_cosThetaL", "gen_cosThetaK", "gen_phi"]].to_numpy(), obs=obs_ang)
        
        print(f"NUMERO DE EVENTOS: {len(obs_Gen_q2)} \n") 

        print("\n" + "="*60)
        print(">>> INICIANDO FIT GEN LEVEL ")
        print("="*60)
        
        result_gen, errors_gen = run_fit(pdf_ang_trans, data_true)
        
        # ======================================================
        # COMPROBACIÓN DE ERRORES (MINOS < 1.0) PARA ESTE BIN
        # ======================================================
        bin_errors_valid = True
        
        if errors_gen:
            for param in fit_params_list:
                if param in errors_gen:
                    param_errs = errors_gen[param]
                    lower_err = abs(param_errs.get('lower', 999.0))
                    upper_err = abs(param_errs.get('upper', 999.0))
                    
                    if lower_err >= 1.0 or upper_err >= 1.0:
                        bin_errors_valid = False
                        print(f"  -> Descartado: {param.name} tiene errores altos (Lower: {lower_err:.4f}, Upper: {upper_err:.4f})")
                        break
                else:
                    bin_errors_valid = False
                    print(f"  -> Descartado: No se encontraron errores MINOS para {param.name}")
                    break
        else:
            bin_errors_valid = False
            print("  -> Descartado: La rutina de cálculo de errores falló completamente.")

        # Si el bin falla, marcamos toda la iteración como inválida y rompemos el bucle de bines
        if not bin_errors_valid:
            print(f">>> Errores inaceptables en {binN}. Descartando la iteración {i} completa.\n")
            iteration_valid = False
            break 
            
        # Si el bin es válido, calculamos y guardamos temporalmente sus valores
        r_values = np.array([result_gen.params[p]['value'] for p in fit_params_list])
        cov_fit = result_gen.covariance(params=fit_params_list)
        
        iteration_results[binN] = {
            'result_gen': result_gen,
            'r_values': r_values,
            'cov_fit': cov_fit
        }

    # ======================================================
    # 3. IMPRESIÓN Y GUARDADO GLOBAL (SÓLO SI TODOS LOS BINS PASARON)
    # ======================================================
    if iteration_valid:
        print(f"\n{'*'*80}")
        print(f"*** AJUSTE EXITOSO EN TODOS LOS BINS PARA LA ITERACIÓN {i}. PROCESANDO RESULTADOS ***")
        print(f"{'*'*80}\n")
        
        for binN, res_data in iteration_results.items():
            result_gen = res_data['result_gen']
            r_values = res_data['r_values']
            cov_fit = res_data['cov_fit']
            
            # Guardado del JSON de zfit
            #save_fit_results(result_gen, binN, base_dir="fit_results/gen_transformed", name=f"fit_results_gen_physical_{binN}")
            
            obs_order = ['FL', 'AFB', 'S3', 'S4', 'S5', 'S7', 'S8', 'S9']
            f_transform = lambda x: get_physical_array(obs_order=obs_order, r_array=x)
            
            # Propagación de covarianza
            J = compute_jacobian(f_transform, r_values)
            cov_phys = J @ cov_fit @ J.T
            errors_phys = np.sqrt(np.diag(cov_phys))
            phys_vals_gen_dict = apply_transformation_equations(*r_values)
            
            print(f"RESUMEN DE OBSERVABLES FÍSICOS (Bin: {binN} | Iteración: {i})")
            print("-"*80)
            print(f"{'Observable':<10} | {'Valor Físico':<15} | {'Error':<25}")
            print("-"*80)

            for j, key in enumerate(obs_order):
                val = phys_vals_gen_dict.get(key, 0.0)
                err = errors_phys[j]
                print(f"{key:<10} | {val:>15.6f} | +/- {err:<15.6f}")
            print("-"*80 + "\n")
    else:
        print(f">>> Saltando la iteración {i} debido a errores en al menos un rango de q2.\n")

Lista de semillas a procesar: [273, 632, 39, 988, 737, 674, 522, 975, 37, 962, 752, 31, 728, 430, 592, 794, 669, 765, 369, 658, 324, 544, 684, 88, 801, 880, 99, 305, 760, 308, 219, 138, 526, 767, 68, 344, 805, 943, 851, 491, 700, 738, 840, 510, 916, 709, 387, 497, 115, 310, 982, 814, 142, 187, 420, 484, 172, 78, 815, 471, 377, 695, 766, 424, 118, 451, 460, 20, 196, 462, 854, 545, 520, 270, 316, 267, 13, 120, 209, 889, 745, 748, 617, 949, 865, 878, 215, 192, 459, 749, 414, 734, 101, 560, 836, 144, 896, 119, 370, 625, 58, 25, 422, 169, 353, 495, 199, 899, 834, 149, 362, 599, 258, 395, 725, 953, 127, 576, 184, 901, 537, 9, 500, 842, 180, 167, 224, 469, 198, 496, 832, 869, 847, 278, 793, 731, 406, 34, 570, 323, 850, 762, 96, 109, 887, 415, 402, 653, 161, 456, 601, 952, 177, 849, 590, 248, 947, 291, 578, 194, 306, 372, 643, 641, 418, 890, 59, 705, 661, 751, 536, 645, 867, 781, 255, 622, 222, 540, 492, 504, 252, 921, 552, 229, 513, 967, 234, 116, 915, 740, 4, 317, 789, 966, 974, 756, 76, 624

2026-03-17 00:09:43.888958: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-17 00:09:43.941682: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-17 00:09:43.943961: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

-> Leyendo semillas iniciales de: fit_results/gen/gen_phy_slsqp/bin10/fit_results_gen_physical_slsqp_bin10.json
NUMERO DE EVENTOS: 18688 


>>> INICIANDO FIT GEN LEVEL 


2026-03-17 00:09:51.779952: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:09:51.944127: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:09:51.953213: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:09:51.962462: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:09:51.986548: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:09:51.995768: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Ch

  -> Descartado: rS4_bin10_iter0 tiene errores altos (Lower: 0.0014, Upper: 2.6000)
>>> Errores inaceptables en bin10. Descartando la iteración 0 completa.

>>> Saltando la iteración 0 debido a errores en al menos un rango de q2.


Iteración: 1 | Procesando bin10, semilla 632 con rango q2: [17.0, 23.0]
-> Leyendo semillas iniciales de: fit_results/gen/gen_phy_slsqp/bin10/fit_results_gen_physical_slsqp_bin10.json
NUMERO DE EVENTOS: 18621 


>>> INICIANDO FIT GEN LEVEL 
  -> Descartado: rS4_bin10_iter1 tiene errores altos (Lower: 0.0934, Upper: 2.3615)
>>> Errores inaceptables en bin10. Descartando la iteración 1 completa.

>>> Saltando la iteración 1 debido a errores en al menos un rango de q2.


Iteración: 2 | Procesando bin10, semilla 39 con rango q2: [17.0, 23.0]
-> Leyendo semillas iniciales de: fit_results/gen/gen_phy_slsqp/bin10/fit_results_gen_physical_slsqp_bin10.json
NUMERO DE EVENTOS: 18550 


>>> INICIANDO FIT GEN LEVEL 


...

2026-03-17 00:14:11.087534: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:14:11.097571: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:14:11.105062: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:14:11.112877: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:14:11.120189: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:14:11.126240: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662601000 = {1, 0} Ch

  -> Descartado: rS4_bin10_iter26 tiene errores altos (Lower: 0.0005, Upper: 2.4809)
>>> Errores inaceptables en bin10. Descartando la iteración 26 completa.

>>> Saltando la iteración 26 debido a errores en al menos un rango de q2.


Iteración: 27 | Procesando bin10, semilla 305 con rango q2: [17.0, 23.0]
-> Leyendo semillas iniciales de: fit_results/gen/gen_phy_slsqp/bin10/fit_results_gen_physical_slsqp_bin10.json
NUMERO DE EVENTOS: 18802 


>>> INICIANDO FIT GEN LEVEL 
  -> Descartado: rS4_bin10_iter27 tiene errores altos (Lower: 0.0011, Upper: 2.6220)
>>> Errores inaceptables en bin10. Descartando la iteración 27 completa.

>>> Saltando la iteración 27 debido a errores en al menos un rango de q2.


Iteración: 28 | Procesando bin10, semilla 760 con rango q2: [17.0, 23.0]
-> Leyendo semillas iniciales de: fit_results/gen/gen_phy_slsqp/bin10/fit_results_gen_physical_slsqp_bin10.json
NUMERO DE EVENTOS: 18910 


>>> INICIANDO FIT GEN LEVEL 
  -> Descartado: rS4_bin10_iter28 tiene errore

2026-03-17 00:17:15.071910: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662600f00 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:17:15.080405: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662600f00 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:17:15.088029: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662600f00 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:17:15.095963: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662600f00 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:17:15.112933: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662600f00 = {1, 0} Check if pdf output contains any NaNs of Infs
2026-03-17 00:17:15.120951: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x72b662600f00 = {1, 0} Ch


Iteración: 44 | Procesando bin10, semilla 916 con rango q2: [17.0, 23.0]
-> Leyendo semillas iniciales de: fit_results/gen/gen_phy_slsqp/bin10/fit_results_gen_physical_slsqp_bin10.json
NUMERO DE EVENTOS: 18949 


>>> INICIANDO FIT GEN LEVEL 
  -> Descartado: rS4_bin10_iter44 tiene errores altos (Lower: 1.8936, Upper: 0.9356)
>>> Errores inaceptables en bin10. Descartando la iteración 44 completa.

>>> Saltando la iteración 44 debido a errores en al menos un rango de q2.


Iteración: 45 | Procesando bin10, semilla 709 con rango q2: [17.0, 23.0]
-> Leyendo semillas iniciales de: fit_results/gen/gen_phy_slsqp/bin10/fit_results_gen_physical_slsqp_bin10.json
NUMERO DE EVENTOS: 18950 


>>> INICIANDO FIT GEN LEVEL 
  -> Descartado: rS4_bin10_iter45 tiene errores altos (Lower: 0.0049, Upper: 2.7397)
>>> Errores inaceptables en bin10. Descartando la iteración 45 completa.

>>> Saltando la iteración 45 debido a errores en al menos un rango de q2.


Iteración: 46 | Procesando bin10, semilla 387

KeyboardInterrupt: 